In [15]:
# Scenario: "The Healthcare Policy Navigator"
# Background
# You are part of the Healthcare Compliance & Policy Team at a large hospital network. New government regulations on patient data privacy and telemedicine practices have just been released. The hospital must quickly adapt to ensure compliance and avoid penalties.
# Challenge
# The hospital uploads a PDF of the healthcare regulation into the Policy Navigator (your Gradio + Chroma + LLM app). Your task is to:
# - Process the regulation document so the assistant can store and understand it.
# - Ask compliance-related questions about patient rights, telemedicine rules, and penalties.
# - Generate clear, actionable answers that can guide doctors, administrators, and IT staff

In [16]:
!pip install gradio pypdf sentence_transformers chromadb mistralai python-dotenv

Defaulting to user installation because normal site-packages is not writeable


In [17]:
# ==========================================================
# HEALTHCARE POLICY NAVIGATOR
# GRADIO + CHROMA + EMBEDDINGS + HUGGINGFACE LLM
# ==========================================================

import gradio as gr
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import chromadb
from mistralai.client import Mistral
from dotenv import load_dotenv
import os
load_dotenv()

True

In [18]:
# ----------------------------------------------------------
# STEP 1 — Load Language Model
# ----------------------------------------------------------

print("Loading LLM...")

llm = Mistral(api_key=os.environ['MISTRAL_API_KEY'])

print("LLM ready")

Loading LLM...
LLM ready


In [19]:
# ----------------------------------------------------------
# STEP 2 — Load Embedding Model
# ----------------------------------------------------------

print("Loading embeddings...")

embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready")

Loading embeddings...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1498.71it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model ready


In [20]:
# ----------------------------------------------------------
# STEP 3 — Initialize Vector Database
# ----------------------------------------------------------

client = chromadb.Client()

try:
    client.delete_collection("policy_docs")
except:
    pass

collection = client.create_collection("policy_docs")

In [21]:
# ----------------------------------------------------------
# STEP 4 — Chunking Function
# ----------------------------------------------------------

def chunk_text(text, chunk_size=600, overlap=80):

    chunks = []
    start = 0

    while start < len(text):

        end = start + chunk_size
        chunk = text[start:end]

        chunks.append(chunk)

        start += chunk_size - overlap

    return chunks

In [22]:
# ----------------------------------------------------------
# STEP 5 — Process Uploaded Policy PDF
# ----------------------------------------------------------

def process_pdf(file):

    reader = PdfReader(file.name)

    text = ""

    for page in reader.pages:

        page_text = page.extract_text()

        if page_text:
            text += page_text

    chunks = chunk_text(text)

    for i, chunk in enumerate(chunks):

        embedding = embedding_model.encode(chunk).tolist()

        collection.add(
            documents=[chunk],
            embeddings=[embedding],
            ids=[str(i)]
        )

    return f"Policy document processed successfully. {len(chunks)} sections indexed."

In [23]:
# ----------------------------------------------------------
# STEP 6 — Retriever
# ----------------------------------------------------------

def retrieve(query, k=3):

    query_embedding = embedding_model.encode(query).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )

    return results["documents"][0]

In [24]:
# ----------------------------------------------------------
# STEP 7 — Generate Compliance Answer
# ----------------------------------------------------------

def answer_question(query):

    docs = retrieve(query)

    context = " ".join(docs)

    prompt = f"""
You are a Healthcare Policy Navigator.

Use the policy context below to answer the question.

Provide a clear and actionable answer that hospital staff
(doctors, administrators, or IT teams) can follow.

Context:
{context}

Question: {query}

Answer:
"""

    response = llm.chat.complete(
        model="mistral-small-latest",
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )
    return response.choices[0].message.content

In [25]:
# ----------------------------------------------------------
# STEP 8 — Chat Function
# ----------------------------------------------------------

def chat(question):

    if not question.strip():
        return "Please enter a compliance question."

    return answer_question(question)

In [26]:
# ----------------------------------------------------------
# STEP 9 — Gradio Interface
# ----------------------------------------------------------

with gr.Blocks() as demo:

    gr.Markdown("# 🏥 Healthcare Policy Navigator")

    gr.Markdown("""
Upload a healthcare regulation document and ask questions about:

• Patient data privacy  
• Telemedicine rules  
• Compliance requirements  
• Penalties for violations  
""")

    pdf_input = gr.File(label="Upload Healthcare Regulation PDF")

    upload_button = gr.Button("Process Policy Document")

    status = gr.Textbox(label="System Status")

    upload_button.click(
        process_pdf,
        inputs=pdf_input,
        outputs=status
    )

    question_box = gr.Textbox(
        label="Ask a Compliance Question"
    )

    answer_box = gr.Textbox(
        label="Policy Guidance",
        lines=12
    )

    ask_button = gr.Button("Get Guidance")

    ask_button.click(
        chat,
        inputs=question_box,
        outputs=answer_box
    )

In [27]:
# ----------------------------------------------------------
# STEP 10 — Launch App
# ----------------------------------------------------------

demo.launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
